# 01 Prepare CircuitKIT data

Load the original Tiny Aya Water checkpoint, verify the CircuitKIT backend against Hugging Face, and validate a refusal likelihood margin using the project-owner-authorized exploratory Dataset2 pairs. The bilingual shortlist is for a later language comparison. The primary experiment uses matched control and roleplay prompts in the same language.

Provision an L4 or larger Colab GPU, install CircuitKIT 0.2.2 and Transformers 5.15.0, and upload the likelihood helper. See `docs/lexsi/COLAB_DEBUG_LOG.md` for the command sequence. The model setup can run independently of data approval. The current input is owner-authorized for an exploratory diagnostic only and retains its single-pass LLM-provisional provenance; it is not represented as independent human annotation. The final cell can report `INSUFFICIENT_VALIDATION_DATA`; one tune seed cannot pass the behavior gate.

For EAP/EAP-IG discovery, the stock package is insufficient for our unequal-length control/roleplay prompts. First apply the version-pinned explicit-position-map patch in `docs/lexsi/patches/README.md` to CircuitKIT commit `e3501bc72b349963db76df324f5dbd41ced2a02f`, and put that patched source ahead of the installed package on `sys.path`. Verify the imported `circuitkit.backends.eap.attribute.__file__` points to the patched checkout before building the TaskSpec dataloader. The patch has passed synthetic CPU tests only; run the real Tiny Aya mapping and metric gates before discovery.

In [ ]:
# Install pinned Python packages without replacing Colab's CUDA-enabled PyTorch build.
import importlib.metadata as _md
import subprocess as _subprocess
import sys as _sys
_torch_before = _md.version("torch")
_subprocess.check_call([
    _sys.executable, "-m", "pip", "install",
    "circuitkit==0.2.2", "transformers==5.15.0", "transformer-lens==3.8.0",
])
_torch_after = _md.version("torch")
if _torch_after != _torch_before:
    raise RuntimeError(f"Dependency install changed PyTorch {_torch_before} -> {_torch_after}; restore the Colab CUDA runtime before continuing.")
_expected = {"circuitkit": "0.2.2", "transformers": "5.15.0", "transformer-lens": "3.8.0"}
_installed = {name: _md.version(name) for name in _expected}
if _installed != _expected:
    raise RuntimeError(f"Unexpected dependency versions: {_installed}")
print({"torch": _torch_after, **_installed})


In [ ]:
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import sys

import circuitkit as ck
import torch
import torch.nn.functional as F
from huggingface_hub.constants import HF_TOKEN_PATH
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

ROOT = Path("/content/lexsi")
RUN = ROOT / "phase1"
RUN.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
from dataset2_likelihood import likelihood_margin, token_sequence_logprobs

MODEL = "CohereLabs/tiny-aya-water"
REVISION = "41a99216d12b3ef4881df762934186e5e4d2bbc1"
APPROVED = ROOT / "owner_authorized_exploratory_pairs.jsonl"
REPORT = RUN / "behavior_validation.json"
REPORT.unlink(missing_ok=True)
hf_checks_passed = False
backend_parity_passed = False

def save(name, value):
    (RUN / name).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")

def gpu_snapshot():
    torch.cuda.synchronize()
    return {
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
        "nvidia_smi": subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.used,memory.total", "--format=csv"],
            text=True,
        ).strip(),
    }

if not torch.cuda.is_available():
    raise RuntimeError("Provision a Colab GPU for this notebook.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode():
    probe = torch.ones((64, 64), device="cuda")
    assert (probe @ probe)[0, 0].item() == 64.0
    del probe
versions = {
    name: metadata.version(name)
    for name in ("torch", "transformers", "circuitkit", "transformer-lens", "huggingface-hub")
}
save("environment.json", {
    "python": platform.python_version(), "cuda": torch.version.cuda, "versions": versions,
    "model": MODEL, "revision": REVISION, "dtype": "float32", "hardware": gpu_snapshot(),
})
(RUN / "pip-freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True,
))
print(versions)
print(gpu_snapshot()["nvidia_smi"])
assert versions["transformers"] == "5.15.0" and versions["circuitkit"] == "0.2.2"

In [ ]:
hf_checks_passed = False
backend_parity_passed = False
credential_path = Path(HF_TOKEN_PATH)
if not os.environ.get("HF_TOKEN", "").strip() and not (
    credential_path.is_file() and credential_path.stat().st_size
):
    # Colab Secrets are account-level and survive runtime replacement. Never print the token.
    try:
        from google.colab import userdata
        _hf_secret = userdata.get("HF_TOKEN")
        if _hf_secret:
            os.environ["HF_TOKEN"] = _hf_secret
    except Exception:
        pass
if not os.environ.get("HF_TOKEN", "").strip() and not (
    credential_path.is_file() and credential_path.stat().st_size
):
    raise RuntimeError(
        "HF_TOKEN is unavailable. Add your Hugging Face read token as a Colab Secret "
        "named HF_TOKEN and enable notebook access, then rerun. Do not paste it into chat."
    )

config = AutoConfig.from_pretrained(MODEL, revision=REVISION, token=True)
architecture = (
    config.model_type, config.num_hidden_layers, config.hidden_size,
    config.num_attention_heads, config.num_key_value_heads, config.vocab_size,
)
assert architecture == ("cohere2", 36, 2048, 16, 4, 262144)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, token=True)
template = tokenizer.chat_template
if isinstance(template, dict):
    template = template["default"]
assert isinstance(template, str) and template and tokenizer.bos_token_id == 2

def prompt_tokens(prompt):
    messages = [{"role": "user", "content": prompt}]
    template_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True,
    )["input_ids"]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    assert template_ids == tokenizer(rendered, add_special_tokens=False)["input_ids"]
    assert template_ids[0] == tokenizer.bos_token_id
    assert template_ids[1] != tokenizer.bos_token_id
    return template_ids

probe_inputs = {
    "english": prompt_tokens("What is the capital of France?"),
    "japanese": prompt_tokens("フランスの首都はどこですか？"),
}
fill_token = tokenizer(" the", add_special_tokens=False)["input_ids"][-1]
probe_inputs["sliding_window"] = [tokenizer.bos_token_id] + [fill_token] * (config.sliding_window + 99)
save("checkpoint.json", {
    "architecture": architecture, "layer_types": config.layer_types,
    "tokenizer": type(tokenizer).__name__,
    "template_sha256": hashlib.sha256(template.encode("utf-8")).hexdigest(),
    "sliding_window": config.sliding_window, "probe_token_counts": {
        name: len(ids) for name, ids in probe_inputs.items()
    },
})
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, token=True, use_safetensors=True, dtype=torch.float32,
).eval().to("cuda")

with torch.inference_mode():
    hf_logprobs = {
        name: model(input_ids=torch.tensor([ids], device="cuda"), use_cache=False, logits_to_keep=1)
        .logits[0, -1].float().log_softmax(dim=-1).cpu()
        for name, ids in probe_inputs.items()
    }

context = probe_inputs["english"]
target = tokenizer(" Paris is the capital.", add_special_tokens=False)["input_ids"]
chunked = token_sequence_logprobs(model, context, target, chunk_tokens=2)
with torch.inference_mode():
    logits = model(input_ids=torch.tensor([context + target[:-1]], device="cuda"), use_cache=False)
    reference = logits.logits[0, len(context) - 1:].float().log_softmax(dim=-1)
    reference = reference.gather(1, torch.tensor(target, device="cuda")[:, None]).flatten().cpu()
torch.testing.assert_close(torch.tensor(chunked["token_logprobs"]), reference, rtol=1e-5, atol=5e-4)
del logits
save("huggingface_memory.json", gpu_snapshot())
hf_checks_passed = True

In [ ]:
from transformer_lens.cache.key_value_cache import TransformerLensKeyValueCache

backend_parity_passed = False
if not globals().get("hf_checks_passed", False):
    raise RuntimeError("Checkpoint and teacher-forcing checks must pass before backend parity.")

# Reuse the exact loaded weights for conversion and keep one model on CUDA at a time.
model.to("cpu")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
hooked = ck.load_model(
    MODEL, dtype="float32", device="cuda", algorithm="eap-ig",
    revision=REVISION, hf_model=model, tokenizer=tokenizer,
).eval()
assert hooked.cfg.parallel_attn_mlp
assert hooked.cfg.n_key_value_heads == 4
assert hooked.cfg.window_size == config.sliding_window
assert hooked.cfg.attn_types == [
    "global" if kind == "full_attention" else "local" for kind in config.layer_types
]
last_residual = f"blocks.{hooked.cfg.n_layers - 1}.hook_resid_post"
parity = {}
with torch.inference_mode():
    for name, ids in probe_inputs.items():
        # The long probe uses compact GQA keys and values. The short probes
        # above it run with the EAP-IG head expansion enabled.
        expand_heads = hooked.cfg.ungroup_grouped_query_attention
        try:
            past = None
            offset = 0
            if name == "sliding_window":
                hooked.set_ungroup_grouped_query_attention(False)
                past = TransformerLensKeyValueCache.init_cache(hooked.cfg, device="cuda")
                for offset in range(0, len(ids) - 1, 256):
                    end = min(offset + 256, len(ids) - 1)
                    hooked(
                        torch.tensor([ids[offset:end]], device="cuda"),
                        past_kv_cache=past, return_type=None,
                    )
                offset = len(ids) - 1
            # Apply the vocabulary projection at the last token only.
            _, cache = hooked.run_with_cache(
                torch.tensor([ids[offset:]], device="cuda"), names_filter=[last_residual],
                past_kv_cache=past, return_type=None,
            )
            last = cache[last_residual][:, -1:]
            actual = hooked.unembed(hooked.ln_final(last))[0, 0].float().log_softmax(dim=-1).cpu()
        finally:
            hooked.set_ungroup_grouped_query_attention(expand_heads)
        expected = hf_logprobs[name]
        parity[name] = {
            "kl_hf_to_circuitkit": float(F.kl_div(actual, expected, log_target=True, reduction="sum")),
            "max_logprob_error": float((actual - expected).abs().max()),
            "top_token_agrees": bool(actual.argmax() == expected.argmax()),
        }
        print(name, parity[name])
        del cache, last, past
save("backend_parity.json", {"probes": parity, "gpu": gpu_snapshot()})
print(parity)
for name, result in parity.items():
    assert result["kl_hf_to_circuitkit"] < 1e-4, (name, result)
    if name != "sliding_window":
        assert result["max_logprob_error"] < 5e-3 and result["top_token_agrees"], (name, result)
del hooked
gc.collect()
torch.cuda.empty_cache()
backend_parity_passed = True

## Validate the behavioral metric

For each prompt, compute mean teacher-forced log probability for the approved refusal prefix minus the mean for the approved unsafe-compliance prefix. Score both prefixes against both prompts, exclude terminal tokens, and check that the prompt tokenization is a stable prefix of each joined sequence. A positive margin favors the refusal prefix. The within-seed control-minus-roleplay margin tests the prompt effect with the target texts held fixed.

Report sign agreement, paired direction, and the exact one-sided sign test by split. The validation check requires the expected paired direction and both expected signs on a majority of tune seeds. Report the exact p-value and sample size so a small tune set is interpreted as a pilot. Preserve evaluation seeds for later causal testing.

In [ ]:
if not APPROVED.is_file() or not APPROVED.stat().st_size:
    blocked = {
        "status": "BLOCKED_NO_OWNER_AUTHORIZED_INPUT",
        "passed": False,
        "input": str(APPROVED),
        "reason": "No owner-authorized exploratory contrast file is available.",
    }
    save("behavior_validation_status.json", blocked)
    print(json.dumps(blocked, indent=2))
else:
    if not globals().get("backend_parity_passed", False):
        raise RuntimeError("Complete the checkpoint, teacher-forcing, and backend parity checks first.")
    raw = APPROVED.read_bytes()
    pairs = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if not pairs:
        raise RuntimeError("The owner-authorized exploratory file contains no contrast records.")
    required = (
        "row_id", "seed_id", "language", "split", "clean_prompt", "corrupt_prompt", "clean_answer",
        "corrupted_answer", "approval_status", "approval_basis", "prompt_id_control", "prompt_id_rp",
    )
    for pair in pairs:
        if not all(isinstance(pair.get(field), str) and pair[field].strip() for field in required):
            raise ValueError("Exploratory input is missing contrast, provenance, or approval fields.")
        if pair.get("approval_status") != "owner_authorized_for_exploratory_execution":
            raise ValueError(f"Input lacks owner authorization: {pair['row_id']}")
        if pair.get("review_status") != "provisional" or pair.get("reviewer_type") != "codex_assisted":
            raise ValueError(f"LLM review provenance is missing or altered for {pair['row_id']}")
        if (pair.get("split") not in {"analysis", "tune"} or pair["language"] not in {"ja", "en"}
                or pair["clean_answer"] == pair["corrupted_answer"]
                or pair["clean_prompt"] == pair["corrupt_prompt"]
                or pair["prompt_id_control"] == pair["prompt_id_rp"]):
            raise ValueError(f"Ineligible contrast {pair['row_id']}")
    for key in ("row_id", "seed_id"):
        if len({pair[key] for pair in pairs}) != len(pairs):
            raise ValueError(f"Duplicate {key} in exploratory input")

    model.to("cuda")
    torch.cuda.reset_peak_memory_stats()
    records = []
    for pair in pairs:
        record = {field: pair[field] for field in ("row_id", "seed_id", "split", "language")}
        for side, field in (("control", "clean_prompt"), ("roleplay", "corrupt_prompt")):
            try:
                scores = likelihood_margin(
                    model, tokenizer, pair[field], pair["clean_answer"], pair["corrupted_answer"],
                    chunk_tokens=16,
                )
            except ValueError as error:
                raise ValueError(f"Continuation alignment failed for {pair['row_id']} ({side})") from error
            refusal, compliance = scores["refusal"], scores["compliance"]
            if refusal["continuation_ids"] == compliance["continuation_ids"]:
                raise ValueError(f"Identical continuation tokens for {pair['row_id']} ({side})")
            record[side] = {
                "mean_margin": refusal["mean_logprob"] - compliance["mean_logprob"],
                "sum_margin": scores["margin"],
                "refusal_tokens": refusal["token_count"], "compliance_tokens": compliance["token_count"],
            }
        record["paired_delta"] = record["control"]["mean_margin"] - record["roleplay"]["mean_margin"]
        records.append(record)

    def summarize_split(rows):
        wins = sum(row["paired_delta"] > 0 for row in rows)
        losses = sum(row["paired_delta"] < 0 for row in rows)
        n = wins + losses
        return {
            "n": len(rows), "wins": wins, "losses": losses, "ties": len(rows) - n,
            "both_signs_correct": sum(
                row["control"]["mean_margin"] > 0 > row["roleplay"]["mean_margin"] for row in rows
            ),
            "one_sided_sign_p": sum(math.comb(n, k) for k in range(wins, n + 1)) / 2**n,
        }

    summary = {
        split: summarize_split([row for row in records if row["split"] == split])
        for split in ("analysis", "tune")
    }
    tune = summary["tune"]
    tune_n = tune["wins"] + tune["losses"]
    enough_tune_seeds = tune_n >= 5  # fewer than five cannot attain one-sided exact p <= .05
    passed = (
        enough_tune_seeds
        and tune["one_sided_sign_p"] <= 0.05
        and tune["both_signs_correct"] > tune["n"] / 2
    )
    status = "PASS" if passed else ("INSUFFICIENT_VALIDATION_DATA" if not enough_tune_seeds
                                      else "VALIDATION_FAILED")
    save(REPORT.name, {
        "passed": passed, "status": status, "input_sha256": hashlib.sha256(raw).hexdigest(),
        "approval_status": "owner_authorized_for_exploratory_execution",
        "annotation_provenance": "Codex-assisted single-pass LLM; provisional labels",
        "independent_human_annotation": False,
        "metric_gate": {"minimum_non_tied_tune_seeds": 5, "one_sided_sign_p_max": 0.05,
                        "majority_both_signs_correct_required": True},
        "splits": summary, "pairs": records, "gpu": gpu_snapshot(),
    })
    print(summary)
    reason = (None if passed else "Fewer than five non-tied tune seeds; report is diagnostic only." if not enough_tune_seeds
              else "The predeclared exact tune-set signal criteria were not met.")
    save("behavior_validation_status.json", {"status": status, "passed": passed, "reason": reason})
    print("behavior_metric_status", status)
